# PRCE-003 – Mall Customers Spending Score Analysis
**Tools:** Python, PySpark, Pandas, Matplotlib

**Assignment tasks:** Load the dataset using PySpark; analyze Spending Score; prepare an analysis and challenges report.

Run cells from top to bottom. The notebook attempts to download the assignment file from Google Drive. If the link is access-restricted, download it manually and place `Mall_Customers.csv` in `../data/raw/` (or upload it when prompted).

## 1. Install dependencies
Run this once if the libraries are not installed in the selected VS Code Python environment.

In [ ]:
%pip install pyspark pandas numpy matplotlib seaborn gdown python-docx

## 2. Imports and paths

In [ ]:
from pathlib import Path
import re
import subprocess
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from pyspark.sql import SparkSession
from pyspark.sql import functions as F

PROJECT_ROOT = Path.cwd()
# If the notebook is opened from the notebooks folder, move to project root.
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIR = PROJECT_ROOT / "data" / "raw"
OUTPUT_DIR = PROJECT_ROOT / "output"
REPORT_DIR = PROJECT_ROOT / "reports"
DATA_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
REPORT_DIR.mkdir(parents=True, exist_ok=True)

DATA_FILE = DATA_DIR / "Mall_Customers.csv"
DRIVE_ID = "1k6A8r1cCdt0Ft0mPtvnYckruaTn2XXz0"
print("Project root:", PROJECT_ROOT)
print("Dataset path:", DATA_FILE)

## 3. Download the assignment dataset
The notebook attempts the provided Google Drive download. If it fails, use the browser link from the assignment brief, download the CSV, and save it at the path shown above.

In [ ]:
if not DATA_FILE.exists():
    try:
        import gdown
        url = f"https://drive.google.com/uc?id={DRIVE_ID}"
        downloaded = gdown.download(url, str(DATA_FILE), quiet=False, fuzzy=True)
        if not downloaded:
            raise RuntimeError("Google Drive download returned no file.")
    except Exception as exc:
        print("Automatic download failed:", exc)
        print("Download manually from:")
        print("https://drive.google.com/file/d/1k6A8r1cCdt0Ft0mPtvnYckruaTn2XXz0/view?usp=share_link")
        print("Save the CSV as:", DATA_FILE)

assert DATA_FILE.exists(), "Dataset CSV not found. Download it manually and place it in data/raw/Mall_Customers.csv."
print("Dataset size (bytes):", DATA_FILE.stat().st_size)

## 4. Start Spark

In [ ]:
spark = (
    SparkSession.builder
    .appName("PRCE-003-Mall-Customers")
    .master("local[*]")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")
print("Spark version:", spark.version)

## 5. Load data using PySpark

In [ ]:
raw_df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(str(DATA_FILE))
)

# Standardize names: e.g. 'Annual Income (k$)' -> 'Annual_Income_k'
df = raw_df
for old_name in raw_df.columns:
    new_name = re.sub(r"[^0-9a-zA-Z]+", "_", old_name.strip()).strip("_")
    df = df.withColumnRenamed(old_name, new_name)

# Accommodate common alternate column names if present.
if "Annual_Income_k_" in df.columns and "Annual_Income_k" not in df.columns:
    df = df.withColumnRenamed("Annual_Income_k_", "Annual_Income_k")
if "Spending_Score_1_100" in df.columns and "Spending_Score" not in df.columns:
    df = df.withColumnRenamed("Spending_Score_1_100", "Spending_Score")

df.show(10, truncate=False)
df.printSchema()
print("Rows:", df.count(), "| Columns:", len(df.columns))
print("Column names:", df.columns)

## 6. Data quality checks

In [ ]:
# Null count per column
null_counts = df.select([
    F.count(F.when(F.col(c).isNull(), c)).alias(c)
    for c in df.columns
])
null_counts.show(truncate=False)

duplicate_count = df.count() - df.dropDuplicates().count()
print("Duplicate rows:", duplicate_count)

# Inspect descriptive statistics for numeric columns.
df.describe().show(truncate=False)

## 7. Prepare Spending Score and calculate summary statistics

In [ ]:
if "Spending_Score" not in df.columns:
    raise ValueError(f"Spending_Score column not found. Available columns: {df.columns}")

analysis_df = df.withColumn("Spending_Score", F.col("Spending_Score").cast("double"))

score_stats = analysis_df.select(
    F.count("Spending_Score").alias("non_null_count"),
    F.round(F.mean("Spending_Score"), 2).alias("mean"),
    F.expr("percentile_approx(Spending_Score, 0.5)").alias("median"),
    F.round(F.stddev("Spending_Score"), 2).alias("stddev"),
    F.min("Spending_Score").alias("minimum"),
    F.max("Spending_Score").alias("maximum")
)
score_stats.show()
score_stats_pd = score_stats.toPandas()
score_stats_pd

## 8. Categorize customers by Spending Score
For this project, use descriptive bands: Low 1–39, Medium 40–60, High 61–100. These cut-offs are project-defined to make the report readable, not official labels.

In [ ]:
scored_df = analysis_df.withColumn(
    "Spending_Group",
    F.when(F.col("Spending_Score").between(1, 39), "Low (1-39)")
     .when(F.col("Spending_Score").between(40, 60), "Medium (40-60)")
     .when(F.col("Spending_Score").between(61, 100), "High (61-100)")
     .otherwise("Outside expected range")
)

group_summary = (
    scored_df.groupBy("Spending_Group")
    .agg(F.count("*").alias("Customer_Count"))
    .withColumn(
        "Percentage",
        F.round(F.col("Customer_Count") * 100.0 / F.sum("Customer_Count").over(
            __import__("pyspark").sql.Window.partitionBy()
        ), 2)
    )
    .orderBy("Spending_Group")
)
group_summary.show(truncate=False)
group_summary_pd = group_summary.toPandas()
group_summary_pd

## 9. Analyze Spending Score by customer segments (if columns exist)

In [ ]:
segment_tables = {}

if "Gender" in scored_df.columns:
    gender_summary = (
        scored_df.groupBy("Gender")
        .agg(
            F.count("*").alias("Customer_Count"),
            F.round(F.avg("Spending_Score"), 2).alias("Average_Spending_Score"),
            F.expr("percentile_approx(Spending_Score, 0.5)").alias("Median_Spending_Score")
        )
        .orderBy("Gender")
    )
    gender_summary.show()
    segment_tables["gender"] = gender_summary.toPandas()

if "Age" in scored_df.columns:
    age_df = scored_df.withColumn(
        "Age_Group",
        F.when(F.col("Age") < 25, "Under 25")
         .when(F.col("Age") < 35, "25-34")
         .when(F.col("Age") < 45, "35-44")
         .when(F.col("Age") < 55, "45-54")
         .otherwise("55+")
    )
    age_summary = (
        age_df.groupBy("Age_Group")
        .agg(
            F.count("*").alias("Customer_Count"),
            F.round(F.avg("Spending_Score"), 2).alias("Average_Spending_Score")
        )
        .orderBy("Age_Group")
    )
    age_summary.show()
    segment_tables["age"] = age_summary.toPandas()

if "Annual_Income_k" in scored_df.columns:
    income_data_pd = scored_df.select("Annual_Income_k", "Spending_Score").toPandas()
    print("Annual income and spending score data prepared for relationship analysis.")

## 10. Visualize Spending Score distribution

In [ ]:
score_pd = scored_df.select("Spending_Score").toPandas()["Spending_Score"].dropna()

plt.figure(figsize=(9, 5))
sns.histplot(score_pd, bins=20, kde=True)
plt.title("Distribution of Mall Customers' Spending Scores")
plt.xlabel("Spending Score")
plt.ylabel("Number of Customers")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "spending_score_distribution.png", dpi=150)
plt.show()

## 11. Visualize Low / Medium / High groups

In [ ]:
plt.figure(figsize=(9, 5))
sns.barplot(data=group_summary_pd, x="Spending_Group", y="Customer_Count")
plt.title("Customers by Spending Score Group")
plt.xlabel("Spending Score Group")
plt.ylabel("Number of Customers")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "spending_score_groups.png", dpi=150)
plt.show()

## 12. Optional segment charts

In [ ]:
if "gender" in segment_tables:
    gender_pd = segment_tables["gender"]
    plt.figure(figsize=(7, 5))
    sns.barplot(data=gender_pd, x="Gender", y="Average_Spending_Score")
    plt.title("Average Spending Score by Gender")
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / "average_spending_by_gender.png", dpi=150)
    plt.show()

if "Annual_Income_k" in scored_df.columns:
    plt.figure(figsize=(8, 5))
    sns.scatterplot(data=income_data_pd, x="Annual_Income_k", y="Spending_Score")
    plt.title("Annual Income vs Spending Score")
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / "income_vs_spending_score.png", dpi=150)
    plt.show()

## 13. Save analysis outputs

In [ ]:
group_summary_pd.to_csv(OUTPUT_DIR / "spending_score_group_summary.csv", index=False)
score_stats_pd.to_csv(OUTPUT_DIR / "spending_score_statistics.csv", index=False)

for segment_name, segment_pd in segment_tables.items():
    segment_pd.to_csv(OUTPUT_DIR / f"spending_by_{segment_name}.csv", index=False)

if "Annual_Income_k" in scored_df.columns:
    income_data_pd.to_csv(OUTPUT_DIR / "income_vs_spending_data.csv", index=False)

print("Saved outputs to:", OUTPUT_DIR)
print("Files:", [p.name for p in OUTPUT_DIR.iterdir()])

## 14. Generate the Word analysis report

In [ ]:
# The report generator reads the saved group summary and creates a Word report.
subprocess.run(
    [sys.executable, str(PROJECT_ROOT / "src" / "generate_report.py")],
    check=True,
    cwd=str(PROJECT_ROOT)
)
print("Word report:", REPORT_DIR / "Mall_Customers_Analysis_Report.docx")

## 15. Challenges faced – personalize before submission
Possible challenges to include only if they genuinely occurred:
- Google Drive access/download permissions.
- Local Java and PySpark configuration in VS Code.
- Standardizing column names and numeric data types.
- Checking null values and duplicate records.
- Selecting and explaining Spending Score bands.

Add what happened, how you diagnosed it, and how you resolved it. Do not claim challenges you did not encounter.

In [ ]:
# Stop Spark when all analysis is complete.
spark.stop()
print("Spark session stopped.")